
## 1. Import required libraries
We will use TensorFlow/Keras for building the LSTM model and NumPy for prediction handling.


In [1]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0



## 2. Create a small text dataset

This is a simple teacher-controlled corpus.  
Students can later add their own sentences and observe how the predictions change.


In [2]:

corpus = [
    "deep learning is powerful",
    "deep learning is fun",
    "machine learning is interesting",
    "artificial intelligence is the future",
    "i love machine learning",
    "lstm is used for sequence prediction",
    "neural networks learn patterns",
    "data science is exciting",
    "python is easy to learn",
    "models improve with more data"
]

print("Number of sentences:", len(corpus))
print("\nSample corpus:")
for line in corpus:
    print("-", line)


Number of sentences: 10

Sample corpus:
- deep learning is powerful
- deep learning is fun
- machine learning is interesting
- artificial intelligence is the future
- i love machine learning
- lstm is used for sequence prediction
- neural networks learn patterns
- data science is exciting
- python is easy to learn
- models improve with more data



## 3. Tokenization

A tokenizer converts words into integers.

Example:
- `deep` → 1
- `learning` → 2

The exact numbers depend on the fitted vocabulary.


In [3]:

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

word_index = tokenizer.word_index
total_words = len(word_index) + 1  # +1 because indexing starts from 1

print("Vocabulary size:", total_words)
print("\nWord index:")
print(word_index)


Vocabulary size: 33

Word index:
{'is': 1, 'learning': 2, 'deep': 3, 'machine': 4, 'learn': 5, 'data': 6, 'powerful': 7, 'fun': 8, 'interesting': 9, 'artificial': 10, 'intelligence': 11, 'the': 12, 'future': 13, 'i': 14, 'love': 15, 'lstm': 16, 'used': 17, 'for': 18, 'sequence': 19, 'prediction': 20, 'neural': 21, 'networks': 22, 'patterns': 23, 'science': 24, 'exciting': 25, 'python': 26, 'easy': 27, 'to': 28, 'models': 29, 'improve': 30, 'with': 31, 'more': 32}


In [4]:

input_sequences = []

for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

print("Total training sequences:", len(input_sequences))
print("\nSome sequences before padding:")
for seq in input_sequences[:10]:
    print(seq)


Total training sequences: 35

Some sequences before padding:
[3, 2]
[3, 2, 1]
[3, 2, 1, 7]
[3, 2]
[3, 2, 1]
[3, 2, 1, 8]
[4, 2]
[4, 2, 1]
[4, 2, 1, 9]
[10, 11]


In [5]:

max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

print("Maximum sequence length:", max_seq_len)
print("\nPadded sequences:")
print(input_sequences[:10])


Maximum sequence length: 6

Padded sequences:
[[ 0  0  0  0  3  2]
 [ 0  0  0  3  2  1]
 [ 0  0  3  2  1  7]
 [ 0  0  0  0  3  2]
 [ 0  0  0  3  2  1]
 [ 0  0  3  2  1  8]
 [ 0  0  0  0  4  2]
 [ 0  0  0  4  2  1]
 [ 0  0  4  2  1  9]
 [ 0  0  0  0 10 11]]



## 6. Split into input (`X`) and output (`y`)

- `X` = all words except the last word
- `y` = the last word to be predicted

We convert `y` into one-hot encoding because this is a multiclass classification problem.


In [6]:

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)


Shape of X: (35, 5)
Shape of y: (35, 33)



## 7. Build the LSTM model

### Model architecture
- **Embedding layer**: converts word indices into dense vectors
- **LSTM layer**: learns sequence patterns and context
- **Dense layer with softmax**: predicts the next word


In [7]:

model = Sequential([
    Embedding(input_dim=total_words, output_dim=10, input_length=max_seq_len - 1),
    LSTM(100),
    Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


## 8. Train the model

Since the dataset is very small, we train for more epochs so the model can learn the patterns better.


In [8]:

history = model.fit(X, y, epochs=200, verbose=1)


Epoch 1/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.0571 - loss: 3.4955
Epoch 2/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.2000 - loss: 3.4894
Epoch 3/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.2000 - loss: 3.4840
Epoch 4/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.2000 - loss: 3.4788
Epoch 5/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.2000 - loss: 3.4734
Epoch 6/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2000 - loss: 3.4667
Epoch 7/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.2000 - loss: 3.4596
Epoch 8/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.2000 - loss: 3.4499
Epoch 9/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2000 - loss: 3.4382
Epoch 10/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2000 - loss: 3.4238
Epoch 11/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2000 - loss: 3.4066
Epoch 12/200
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.2000 - lo

In [9]:

def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')

        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        seed_text += " " + output_word
    return seed_text



## 10. Test the model

Try a few seed texts and observe the output.


In [10]:

print(generate_text("deep learning", 2))
print(generate_text("machine learning", 2))
print(generate_text("python is", 2))


deep learning is fun
machine learning is fun
python is exciting to
